# Phase 3: Molecular Graph Neural Networks (DeepChem ESOL)
**Topic**: DeepChem Graph Convolutional Neural Networks (`GraphConvModel`)  
**Resource**: [DeepChem Official Tutorials](https://deepchem.io/) (Tutorials 1–5)

This notebook featurizes small molecules from SMILES strings into `ConvMol` graph representations and trains a DeepChem `GraphConvModel` to predict log aqueous solubility (log S), evaluating performance against the Phase 1 Random Forest baseline.

In [ ]:
# Install dependencies for Google Colab environment
!pip install deepchem rdkit scikit-learn pandas numpy matplotlib

In [ ]:
import deepchem as dc
from deepchem.feat import ConvMolFeaturizer
from deepchem.models import GraphConvModel
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Fetch Delaney Dataset & Featurize as ConvMol Graphs
url = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
df = pd.read_csv(url)
print(f"Loaded {len(df)} compounds from Delaney dataset.")

featurizer = ConvMolFeaturizer()
features = featurizer.featurize(df["smiles"].tolist())
targets = df["measured log solubility in mols per litre"].to_numpy()

dataset = dc.data.NumpyDataset(X=features, y=targets, ids=df["smiles"].tolist())
splitter = dc.splits.RandomSplitter()
train_dataset, test_dataset = splitter.train_test_split(dataset, frac_train=0.8, seed=42)
print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

In [ ]:
# 2. Build & Train DeepChem GraphConvModel
model = GraphConvModel(
    n_tasks=1,
    mode='regression',
    dropout=0.2,
    dense_layer_size=128,
    graph_conv_layers=[128, 128],
    random_seed=42
)

print('Training GraphConvModel...')
model.fit(train_dataset, nb_epoch=60)
print('Training Complete.')

In [ ]:
# 3. Evaluate Metrics on Independent Test Set
test_y_true = test_dataset.y
test_y_pred = model.predict(test_dataset).flatten()

r2 = r2_score(test_y_true, test_y_pred)
rmse = np.sqrt(mean_squared_error(test_y_true, test_y_pred))

print(f"GraphConv Test R² Score : {r2:.4f}")
print(f"GraphConv Test RMSE     : {rmse:.4f}")

In [ ]:
# 4. Plot Measured vs Predicted Solubility
plt.figure(figsize=(7, 6))
plt.scatter(test_y_true, test_y_pred, alpha=0.75, color="#10B981", edgecolors="k", linewidth=0.5)
plt.plot([test_y_true.min(), test_y_true.max()], [test_y_true.min(), test_y_true.max()], "r--", label="Ideal 1:1")
plt.xlabel("Measured log S (mol/L)")
plt.ylabel("Predicted log S (mol/L)")
plt.title(f"DeepChem GraphConvModel (ESOL Solubility)\nTest R² = {r2:.3f}, RMSE = {rmse:.3f}")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()